# GISLR — semantically duplicate labels: which to merge, and what it's worth

**Standalone diagnostic** (not a pipeline stage): reads the run registry and
`sign_to_prediction_index_map.json`, writes `sb.recognize.label_merge` (the decided merge
groups, already in `packages/`) and this notebook's own evidence tables into
`docs/reports/label-merging.md`. Nothing here changes a run record, a checkpoint, or the
dataset's labels.

**The user's task (2026-09-25): "analyze all the glosses of gislr; if some labels are
semantically similar then make them into the same label, as the model is mislabelling
them; then check the accuracy of the top performing models."** Two parts:

1. **§1-§3**: a systematic pass over all 250x249/2 gloss pairs (not just a hand-picked
   list) for semantic similarity, cross-referenced against the aggregate confusion
   matrix — a pair is only merged when **both** agree. `docs/reports/
   plateau-diagnosis.md` §1 already found and named 14 of these pairs from a 16-pair
   hand list (pre-registry-reset, old split); this notebook re-derives the set from
   scratch on the *current* canonical split, checks nothing was missed, and packages
   the result as reusable code (`sb.recognize.label_merge`) instead of a one-off table.
2. **§4**: strict vs. merged accuracy for the current top canonical models. **Blocked on
   `gislr.1.models.training.ipynb`'s run in progress (TODO §4.3's 12-run grid) — the
   user asked to wait for it before this step.** §4's cells are written and will run
   once those runs are evaluated; until then they report on whatever is on the
   leaderboard *now*, clearly labeled as provisional.

## Why WordNet alone doesn't decide this

Wu-Palmer similarity (taxonomic tree distance) rates sibling categories — `cat`/`dog`,
any two colors, `horse`/`zebra` — just as high as true near-synonyms, because it measures
"same branch of the tree," not "same meaning." §1 shows this concretely. It's supporting
evidence, cross-checked against §2's empirical confusion, never the sole decision.

## 1. Semantic similarity, systematically — and why it needs a second check

Dominant WordNet sense per part of speech (the *first*, most-frequent synset — not
the best-scoring sense across all of a word's meanings, which turns out to match
coincidentally through obscure senses for short common words: naive best-over-all-senses
scoring puts `make`/`pretend` and `stay`/`stuck` at a perfect 1.0 through senses no
GISLR video means). Even restricted to dominant senses, Wu-Palmer similarity clusters
same-category siblings as tightly as real synonyms — the table below shows both kinds
side by side; §2 is what tells them apart.

In [1]:
# ===== Imports, vocab, WordNet setup =====
import itertools
import json
from pathlib import Path

import nltk
import numpy as np
import pandas as pd

from sb.core.paths import CACHE_DIR, EXTERNAL_DIR, MODELS_DIR, ROOT_DIR, gislr_dir
from sb.core.vocab import load_label_map
from sb.mlops import query as SUB
from sb.recognize import label_merge as LM

DATA_DIR = gislr_dir()
LABEL_MAP = load_label_map(DATA_DIR)
WORDS = sorted(LABEL_MAP, key=lambda w: LABEL_MAP[w])
assert len(WORDS) == 250
EVAL_DIR = CACHE_DIR / "gislr" / "evaluation"

nltk_root = EXTERNAL_DIR / "nltk_data"
if str(nltk_root) not in nltk.data.path:
    nltk.data.path.append(str(nltk_root))
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk_root.mkdir(parents=True, exist_ok=True)
    nltk.download("wordnet", download_dir=str(nltk_root), quiet=True)
    nltk.download("omw-1.4", download_dir=str(nltk_root), quiet=True)
from nltk.corpus import wordnet as wn

print(f"{len(WORDS)} glosses, label map from {DATA_DIR.name}")

250 glosses, label map from 1


In [2]:
# ===== Dominant-sense WordNet similarity over every pair =====
# Compound/non-dictionary GISLR tokens mapped to a lookupable lemma; pronouns and other
# purely grammatical tokens get None (no WordNet entry -- and none should be invented).
LOOKUP = {
    "callonphone": "call", "frenchfries": "french_fries", "glasswindow": "window",
    "icecream": "ice_cream", "hesheit": None, "minemy": None, "weus": None,
    "haveto": "have", "yourself": None, "TV": "television",
}


def dominant_synsets(w: str):
    """The single most frequent synset per part of speech (WordNet lists senses by
    frequency, so [0] is the dominant one) -- not the best match across every sense."""
    lw = LOOKUP.get(w, w)
    if lw is None:
        return []
    by_pos: dict[str, object] = {}
    for s in wn.synsets(lw.replace(" ", "_")):
        by_pos.setdefault(s.pos(), s)  # first hit per POS = dominant
    return list(by_pos.values())


SYNSETS = {w: dominant_synsets(w) for w in WORDS}


def wn_similarity(a: str, b: str) -> tuple[float, str]:
    sa, sb = SYNSETS[a], SYNSETS[b]
    best, why = 0.0, ""
    for s1 in sa:
        for s2 in sb:
            if s1.pos() != s2.pos():
                continue
            if s1 == s2:
                return 1.0, f"same dominant synset {s1.name()}"
            sim = s1.wup_similarity(s2) or 0.0
            if sim > best:
                best, why = sim, f"{s1.name()} ~ {s2.name()}"
    return best, why


rows = []
for a, b in itertools.combinations(WORDS, 2):
    sim, why = wn_similarity(a, b)
    if sim >= 0.75:
        rows.append({"a": a, "b": b, "wn_sim": round(sim, 3), "why": why})
wn_df = pd.DataFrame(rows).sort_values("wn_sim", ascending=False).reset_index(drop=True)
print(f"{len(wn_df)} pairs at wn_sim >= 0.75 (dominant-sense Wu-Palmer)")
wn_df.head(20)

233 pairs at wn_sim >= 0.75 (dominant-sense Wu-Palmer)


,a,b,wn_sim,why
0,after,later,1.000,same dominant synset subsequently.r.01
1,have,haveto,1.000,same dominant synset rich_person.n.01
2,dry,dryer,1.000,same dominant synset dry.a.01
3,cereal,grass,0.957,cereal.n.01 ~ grass.n.01
4,bedroom,room,0.941,bedroom.n.01 ~ room.n.01
5,horse,zebra,0.933,horse.n.01 ~ zebra.n.01
6,duck,goose,0.929,duck.n.01 ~ goose.n.01
7,dog,wolf,0.929,dog.n.01 ~ wolf.n.01
8,doll,toy,0.923,doll.n.01 ~ plaything.n.01
9,airplane,helicopter,0.917,airplane.n.01 ~ helicopter.n.01


Confirms the problem: `horse`/`zebra` (0.93), `dog`/`wolf` (0.93), every color pair
(0.875), `brother`/`uncle` (0.857) all score as high as `pen`/`pencil` (0.889) or
`nap`/`sleep` (0.889) — taxonomic closeness, not synonymy. On to §2.

## 2. Cross-reference with the *current* aggregate confusion matrix

`data/cache/gislr/evaluation/confusion_all_normalized.npy` already exists but predates
today's training run and may mix in pre-reset (legacy-split) runs — rebuild it here from
every **current-split** canonical run on the leaderboard right now, the same recipe as
`gislr.2.models.evaluation.ipynb`. This is the *primary* evidence: a pair only gets
merged if the model actually spends accuracy confusing it, not because English makes
them sound alike.

In [3]:
# ===== Rebuild the confusion matrix from current-split canonical runs only =====


def confusion_matrix(run_dir: Path, n_classes: int = 250, normalize: bool = True):
    npz_path = run_dir / "assets" / "val_predictions.npz"
    d = np.load(npz_path)
    cm = np.zeros((n_classes, n_classes), dtype=np.float64)
    np.add.at(cm, (d["labels"].astype(int), d["preds"].astype(int)), 1)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)
    return cm


board = SUB.leaderboard("gislr", include_legacy=False)
evaluated = board[board["eval_status"] == "canonical"]
mats, used = [], []
for rid in evaluated["run_id"]:
    p = MODELS_DIR / str(int(rid)) / "assets" / "val_predictions.npz"
    if p.is_file():
        mats.append(confusion_matrix(MODELS_DIR / str(int(rid))))
        used.append(int(rid))
assert mats, "no current-split canonical runs with cached predictions"
cm_current = np.mean(mats, axis=0)
print(f"{len(used)} current-split canonical runs averaged: {used}")

wi = {w: i for i, w in enumerate(WORDS)}


def symmetric_rate(a: str, b: str) -> tuple[float, float, float]:
    ia, ib = wi[a], wi[b]
    return cm_current[ia, ib], cm_current[ib, ia], (cm_current[ia, ib] + cm_current[ib, ia]) / 2


wn_df[["a_to_b", "b_to_a", "mean_rate"]] = wn_df.apply(
    lambda r: pd.Series(symmetric_rate(r["a"], r["b"])), axis=1)
wn_df = wn_df.sort_values("mean_rate", ascending=False)
print(f"\ntop candidates by empirical confusion (of the {len(wn_df)} semantically-scored pairs):")
wn_df.head(25)

7 current-split canonical runs averaged: [1789559734, 1789558839, 1789560829, 1790143122, 1790144582, 1790146838, 1790142624]

top candidates by empirical confusion (of the 233 semantically-scored pairs):


,a,b,wn_sim,why,a_to_b,b_to_a,mean_rate
25,pen,pencil,0.889,pen.n.01 ~ pencil.n.01,0.203571,0.208791,0.206181
6,duck,goose,0.929,duck.n.01 ~ goose.n.01,0.174603,0.184448,0.179526
44,cut,scissors,0.857,cut.v.01 ~ scissor.v.01,0.129344,0.152510,0.140927
116,animal,have,0.800,animal.n.01 ~ rich_person.n.01,0.148980,0.117271,0.133125
2,dry,dryer,1.000,same dominant synset dry.a.01,0.108059,0.133205,0.120632
22,nap,sleep,0.889,nap.v.01 ~ sleep.v.01,0.150183,0.043233,0.096708
18,dog,puppy,0.897,dog.n.01 ~ puppy.n.01,0.037594,0.120724,0.079159
4,bedroom,room,0.941,bedroom.n.01 ~ room.n.01,0.131868,0.009653,0.070760
68,bird,goose,0.833,bird.n.01 ~ goose.n.01,0.038801,0.072333,0.055567
14,rain,snow,0.900,rain.n.01 ~ snow.n.01,0.035714,0.020408,0.028061


## 3. All top empirical confusions, whether or not WordNet flagged them

§1's Wu-Palmer scan only catches pairs *WordNet itself* thinks are related — it will
miss real synonym pairs WordNet's dominant sense doesn't line up with (already known
casualties: `give`/`gift` needed the noun/verb cross, `awake`/`wake` matched through an
adverb sense). So separately: every pair with real empirical confusion, semantically
judged by hand (the only reliable arbiter here — see the module docstring for why), not
filtered by §1's score.

In [4]:
# ===== Every pair with real empirical confusion (mean_rate > 0.03), independent of §1 =====
all_rows = []
for i in range(250):
    for j in range(i + 1, 250):
        r = (cm_current[i, j] + cm_current[j, i]) / 2
        if r > 0.03:
            all_rows.append({"a": WORDS[i], "b": WORDS[j], "mean_rate": round(r, 4),
                             "a_to_b": round(cm_current[i, j], 4), "b_to_a": round(cm_current[j, i], 4)})
confused_df = pd.DataFrame(all_rows).sort_values("mean_rate", ascending=False).reset_index(drop=True)
print(f"{len(confused_df)} pairs with mean confusion > 0.03")
confused_df

134 pairs with mean confusion > 0.03


,a,b,mean_rate,a_to_b,b_to_a
0,awake,wake,0.3652,0.3589,0.3714
1,pen,pencil,0.2062,0.2036,0.2088
2,lips,mouth,0.2031,0.2011,0.2051
3,duck,goose,0.1795,0.1746,0.1844
4,stay,that,0.1541,0.1371,0.1711
...,...,...,...,...,...
129,vacuum,zipper,0.0306,0.0422,0.0190
130,cow,cowboy,0.0303,0.0229,0.0376
131,clown,mad,0.0302,0.0366,0.0238
132,cute,red,0.0301,0.0094,0.0508


### Verdict (hand-judged against the table above; full rationale in each group's `note`)

**Merge — semantically the same/near-synonymous concept, and empirically confused**
(now `sb.recognize.label_merge.MERGE_GROUPS`):

| group | canonical | mean rate(s) |
|---|---|---|
| `awake` <-> `wake` | `wake` | 0.356 |
| `give` <-> `gift` | `gift` | 0.161 |
| `listen` <-> `hear` | `listen` | 0.181 |
| `nap` <-> `sleep` <-> `sleepy` | `sleep` | 0.116 / 0.118 / 0.049 |
| `kitty` <-> `cat` | `cat` | 0.124 |
| `pencil` <-> `pen` | `pencil` | 0.182 |
| `look` <-> `see` | `see` | 0.080 |
| `mouth` <-> `lips` | `lips` | 0.224 |
| `puppy` <-> `dog` | `dog` | 0.077 |

**Considered, empirically confused, but rejected — not a semantic duplicate** (a tool and
its characteristic action, a compound sign sharing one component, or unrelated meanings
that likely share a handshape): `cut`/`scissors`, `goose`/`duck`, `bedroom`/`bed`,
`bedroom`/`room`, `dryer`/`dry`, `hear`/`ear`, `wait`/`finger`, `touch`/`find`,
`please`/`minemy`, `bad`/`thankyou`, `stay`/`that`, `animal`/`have`, `animal`/`bath`.
Matches `docs/reports/plateau-diagnosis.md` §1's independent read of `finger`/`wait` and
`animal`/`have` as "not semantically related."

**Considered, semantically plausible, but rejected — the model already tells them apart**
(near-zero in the current matrix, so merging would just remove a real distinction):
`not`/`no`, `cute`/`pretty`, `shower`/`bath`, `talk`/`say`, `fall`/`drop`, `bad`/`yucky`,
`loud`/`noisy`, `later`/`after`, `that`/`there`, `quiet`/`shhh` (weak: 0.026 mean).

**Same-category siblings, correctly not flagged as confused by the model, so not
merged**: `bug`/`bee`, `lion`/`tiger`, `wolf`/`dog`, `horse`/`donkey`, and every color/
animal/furniture/clothing pair §1 scored highly by pure taxonomy.

In [5]:
# ===== Verify the empirical table above against sb.recognize.label_merge (no drift) =====
group_words = LM.word_to_canonical()
covered = confused_df[confused_df.apply(
    lambda r: r["a"] in group_words or r["b"] in group_words, axis=1)]
print("Confused pairs (mean_rate > 0.03) touching a merge group:")
print(covered[["a", "b", "mean_rate"]].to_string(index=False))
print(f"\n{len(LM.MERGE_GROUPS)} merge groups, {len(group_words)} words folded, "
      f"canonical labels: {sorted({g.canonical for g in LM.MERGE_GROUPS})}")

Confused pairs (mean_rate > 0.03) touching a merge group:
     a      b  mean_rate
 awake   wake     0.3652
   pen pencil     0.2062
  lips  mouth     0.2031
  hear listen     0.1436
 sleep sleepy     0.1338
  gift   give     0.1326
   cat  kitty     0.1323
   nap  sleep     0.0967
   ear   hear     0.0917
   dog  puppy     0.0792
  look    see     0.0711
 mouth  tooth     0.0630
  lips  tooth     0.0570
   nap sleepy     0.0522
pretty  sleep     0.0481
   pen  potty     0.0457
 kitty    see     0.0449

9 merge groups, 19 words folded, canonical labels: ['cat', 'dog', 'gift', 'lips', 'listen', 'pencil', 'see', 'sleep', 'wake']


## 4. Strict vs. merged accuracy for the top canonical models

**Blocked on `gislr.1.models.training.ipynb`'s run in progress (TODO §4.3) — the
user asked to wait for it, then evaluate, then run this section.** Written and ready;
re-run after `gislr.2.models.evaluation.ipynb` backfills the new runs' `val_predictions.
npz`. Reports on the leaderboard *as it stands right now* so the cell is exercised, but
that ranking is provisional — §4.3's 12 pending runs are expected to change it.

In [6]:
# ===== Top-N current-split canonical models: strict vs. merged accuracy =====
TOP_N = 10

top = evaluated.sort_values("accuracy", ascending=False).head(TOP_N)
rows = []
for _, run in top.iterrows():
    run_dir = MODELS_DIR / str(int(run["run_id"]))
    npz_path = run_dir / "assets" / "val_predictions.npz"
    if not npz_path.is_file():
        continue
    d = np.load(npz_path)
    r = LM.merge_report(d["labels"], d["preds"], LABEL_MAP)
    rows.append({"run_id": int(run["run_id"]), "architecture": run["architecture"],
                "landmark_subset": run.get("landmark_subset"), **r})
merge_table = pd.DataFrame(rows)
print(f"top {len(merge_table)} current-split canonical models "
      f"(provisional -- TODO section 4.3's grid is still training):")
merge_table[["run_id", "architecture", "landmark_subset", "strict_accuracy", "merged_accuracy", "delta"]]

top 7 current-split canonical models (provisional -- TODO section 4.3's grid is still training):


,run_id,architecture,landmark_subset,strict_accuracy,merged_accuracy,delta
0,1789559734,gru,None,0.751746,0.765665,0.013918
1,1789558839,gru,None,0.745025,0.759102,0.014077
2,1789560829,gru,None,0.742485,0.757250,0.014765
3,1790143122,gru_continuous,None,0.718776,0.732430,0.013654
4,1790144582,lstm_continuous,None,0.714437,0.729678,0.015241
5,1790146838,gru_continuous,None,0.669560,0.685383,0.015823
6,1790142624,gru_continuous,None,0.335415,0.341977,0.006562


## 5. Write the report

Regenerates `docs/reports/label-merging.md`'s "Current numbers" section from the tables
above — run after §4 has the final (post-training) top models.

In [7]:
# ===== Write the current-numbers block into the report =====
report_path = ROOT_DIR / "docs" / "reports" / "label-merging.md"
if report_path.is_file() and not merge_table.empty:
    block = ["## Current numbers (regenerated by this notebook)\n\n",
             f"{len(used)} current-split canonical runs in the aggregate confusion matrix: {used}.\n\n",
             "| run | architecture | subset | strict | merged | delta |\n",
             "|---|---|---|---|---|---|\n"]
    for _, r in merge_table.iterrows():
        block.append(f"| `{r['run_id']}` | {r['architecture']} | {r['landmark_subset']} | "
                      f"{r['strict_accuracy']:.4f} | {r['merged_accuracy']:.4f} | "
                      f"+{r['delta']:.4f} |\n")
    text = report_path.read_text(encoding="utf-8")
    marker = "## Current numbers (regenerated by this notebook)"
    if marker in text:
        text = text[: text.index(marker)] + "".join(block)
    else:
        text = text.rstrip() + "\n\n" + "".join(block)
    report_path.write_text(text, encoding="utf-8")
    print(f"wrote {report_path}")
else:
    print("skipped: report doesn't exist yet, or section 4 has no rows (training still in progress)")

wrote C:\Users\user2\repository\docs\reports\label-merging.md
